# Numerical Integration and Visualization

This notebook demonstrates numerical integration techniques and visualization of results.

## Problem Statement
We solve the initial value problem:
$$\frac{dy}{dt} = -2y + \sin(t), \quad y(0) = 1.0$$

The analytical solution is:
$$y(t) = \frac{2\cos(t) + \sin(t)}{5} + \frac{3}{5}e^{-2t}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, odeint
from typing import Callable, Tuple

# Define the ODE system
def ode_system(t: float, y: np.ndarray) -> np.ndarray:
    """
    dy/dt = -2y + sin(t)
    """
    return -2.0 * y + np.sin(t)

# Analytical solution
def analytical_solution(t: np.ndarray) -> np.ndarray:
    """
    y(t) = (2*cos(t) + sin(t))/5 + (3/5)*exp(-2*t)
    """
    return (2.0 * np.cos(t) + np.sin(t)) / 5.0 + (3.0 / 5.0) * np.exp(-2.0 * t)

# Initial condition
y0 = np.array([1.0])
t_span = (0.0, 5.0)
t_eval = np.linspace(0.0, 5.0, 200)

print(f"Initial condition: y(0) = {y0[0]:.4f}")
print(f"Integration interval: [{t_span[0]}, {t_span[1]}]")
print(f"Number of evaluation points: {len(t_eval)}")

In [ ]:
# Solve using RK45 (adaptive Runge-Kutta 4th/5th order)
sol_rk45 = solve_ivp(
    ode_system,
    t_span,
    y0,
    method="RK45",
    t_eval=t_eval,
    dense_output=True,
    rtol=1e-8,
    atol=1e-10,
)

print(f"RK45 Status: {sol_rk45.status}")
print(f"RK45 Number of steps: {len(sol_rk45.t)}")
print(f"RK45 Final value: y({sol_rk45.t[-1]:.2f}) = {sol_rk45.y[0, -1]:.6f}")

In [ ]:
# Solve using DOP853 (8th order)
sol_dop = solve_ivp(
    ode_system,
    t_span,
    y0,
    method="DOP853",
    t_eval=t_eval,
    rtol=1e-9,
    atol=1e-11,
)

print(f"DOP853 Status: {sol_dop.status}")
print(f"DOP853 Number of steps: {len(sol_dop.t)}")

In [ ]:
# Compute analytical solution and errors
y_analytical = analytical_solution(t_eval)

error_rk45 = np.abs(sol_rk45.y[0, :] - y_analytical)
error_dop = np.abs(sol_dop.y[0, :] - y_analytical)

print(f"RK45 Max error: {np.max(error_rk45):.2e}")
print(f"RK45 Mean error: {np.mean(error_rk45):.2e}")
print()
print(f"DOP853 Max error: {np.max(error_dop):.2e}")
print(f"DOP853 Mean error: {np.mean(error_dop):.2e}")

## Visualization

We compare the numerical and analytical solutions.

In [ ]:
# Create figure with 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Subplot 1: Solutions
ax = axes[0, 0]
ax.plot(t_eval, y_analytical, 'k-', linewidth=2.0, label='Analytical')
ax.plot(sol_rk45.t, sol_rk45.y[0, :], 'b.', markersize=4.0, label='RK45')
ax.plot(sol_dop.t, sol_dop.y[0, :], 'r+', markersize=5.0, label='DOP853')
ax.set_xlabel('t', fontsize=12)
ax.set_ylabel('y(t)', fontsize=12)
ax.set_title('Numerical vs Analytical Solution', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Subplot 2: RK45 Error
ax = axes[0, 1]
ax.semilogy(t_eval, error_rk45, 'b-', linewidth=1.5, label='RK45')
ax.set_xlabel('t', fontsize=12)
ax.set_ylabel('Absolute Error', fontsize=12)
ax.set_title('RK45 Error vs Time', fontsize=13)
ax.grid(True, alpha=0.3)

# Subplot 3: DOP853 Error
ax = axes[1, 0]
ax.semilogy(t_eval, error_dop, 'r-', linewidth=1.5, label='DOP853')
ax.set_xlabel('t', fontsize=12)
ax.set_ylabel('Absolute Error', fontsize=12)
ax.set_title('DOP853 Error vs Time', fontsize=13)
ax.grid(True, alpha=0.3)

# Subplot 4: Error comparison (log scale)
ax = axes[1, 1]
ax.semilogy(t_eval, error_rk45, 'b-', linewidth=1.5, label='RK45')
ax.semilogy(t_eval, error_dop, 'r-', linewidth=1.5, label='DOP853')
ax.set_xlabel('t', fontsize=12)
ax.set_ylabel('Absolute Error', fontsize=12)
ax.set_title('Error Comparison (log scale)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("Plot rendered successfully.")

In [ ]:
# Summary statistics
summary = {
    "analytical_y_final": float(y_analytical[-1]),
    "rk45_y_final": float(sol_rk45.y[0, -1]),
    "dop_y_final": float(sol_dop.y[0, -1]),
    "rk45_max_error": float(np.max(error_rk45)),
    "dop_max_error": float(np.max(error_dop)),
    "rk45_steps": len(sol_rk45.t),
    "dop_steps": len(sol_dop.t),
}

print("\nSummary Statistics:")
print("-" * 50)
for key, value in summary.items():
    print(f"{key:25s}: {value}")